In [1]:
# Una delle versioni migliori in giornata 12/05
# Feature extraction con CSP, sliding window e classificazione binaria (riposo vs attivazione). 
# Classificatore SVM con gridSearch.

# Il giorno 2/06 ho ripreso in mano il progetto: primo step, aggiungere logica probabilistica alla svm
import mne
from mne.decoding import CSP
from mne_bids import BIDSPath, read_raw_bids
import numpy as np
from sklearn import preprocessing
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from pathlib import Path
from util.preprocessing import create_sliding_windows
from util.preprocessing import create_window_labels
from util.DualBandCSP import DualBandCSP
import importlib


importlib.reload(preprocessing) # Necessario per ricaricare preprocessing se ha subito modifiche
mne.set_log_level('WARNING')

root = "../data"
root = (Path(root).resolve())
runs = ["4", "8", "12"]   # Le run che vengono prese in considerazione
train_runs = [runs[1], runs[2]]
test_run = runs[0] 
first_person = 1
people = 10
all_accuracy = np.zeros(people)  # Per memorizzare l'accuratezza di ogni soggetto
cm_sum = np.zeros((2,2))            # Media delle matrici di confusione

# Eseguiamo la scansione di tutti i soggetti 
for i in range(first_person, first_person + people):
    subject = f"{i:03d}"
    X_train = []
    y_train = []

    X_test = []
    y_test = []

    # Per ogni soggetto eseguiamo la scansione sulle run di nostro interesse
    for run in runs:
        bids_path = BIDSPath(   # Specifichiamo il percorso del dataset e BIDS eseguirà correttamente la scansione 
            subject=subject,
            task="motion",
            run=run,
            datatype="eeg",
            root=root,
        )
        
        try:
            # Fase 1: lettura dei dati
            raw = read_raw_bids(bids_path, verbose=False)  
            events, event_id = mne.events_from_annotations(raw, verbose=False) 
            raw.load_data(verbose=False) # Carico i dati in memoria per poter filtrare ecc.
            raw.filter(l_freq=8, h_freq=30, verbose=False)  # Filtro passa banda
            raw.set_eeg_reference('average', projection=False, verbose=False)  # Riferimento medio
            
            # Applico ICA su tutti i canali
            ica = mne.preprocessing.ICA(n_components=15, method='fastica', random_state=42, verbose=False)
            ica.fit(raw, verbose=False)
            eog_indices, _ = ica.find_bads_eog(raw, ch_name=["Fp1", "Fp2"], verbose=False)
            muscle_indices, _ = ica.find_bads_muscle(raw, verbose=False)
            ica.exclude = list(set(eog_indices + muscle_indices))
            ica.apply(raw, verbose=False)
            # print(f"Soggetto {subject} run {run} - Componenti rimosse: {len(ica.exclude)} {ica.exclude}")

            event_map = {
                event_id['TASK2T0']: 1,
                event_id['TASK2T1']: 2,
                event_id['TASK2T2']: 3
            }
            sfreq = raw.info['sfreq']  # frequenza di campionamento

            # Fase 2: pre-processing
            window_size = 1  # Lunghezza finestra in secondi
            step_size = 0.5  # Lunghezza passo in secondi

            windows, window_samples, step_samples, total_samples = create_sliding_windows(
                raw,
                window_size,
                step_size
            )        

            C_channels = ["C1", "C2", "C3", "C4", "C5", "C6", "Cz"]
            CP_channels = [ch for ch in raw.ch_names if ch.startswith("Cp")]
            FC_channels = [ch for ch in raw.ch_names if ch.startswith("Fc")]
            T_channels =  [ch for ch in raw.ch_names if ch.startswith("T")]
            F_channels =  ["F1", "F2", "F3", "F4", "F5", "F6"]
            P_channels =  ["P1", "P2", "P3", "P4", "P5", "P6"]
            channels_of_interest = C_channels  + FC_channels + CP_channels 
            channels_of_interest = ["C3", "C4", "Cz", "Fc3", "Fc4", "Fcz", "Cp3", "Cp4", "Cpz"]
            picks = mne.pick_channels(raw.ch_names, channels_of_interest)
            windows = windows[:, picks, :]

            y = create_window_labels(
                events,
                event_map,
                total_samples,
                window_samples,
                step_samples,
                threshold= 0.8
            )
            
            # Divido la classificazione in due step: prima distinguo tra stato di riposo e di attivazione.
            # In caso di attivazione, distinguo tra sinistra e destra. Per ora faccio solo la prima parte.
            y_rest_active = np.where(y == 1, 0, 1)  
             
            # Variabile per addestrare il modello sul sinistra/destra (da commentare se non si vuole usare)
            mask_active = y != 1
            y_active = y[mask_active]
            y_lr = np.where(y_active == 2, 0, 1)

            # Assegno a y_binary le etichette desiderate (se voglio rest/active commento la seconda riga)
            y_binary = y_lr
            windows = windows[mask_active]
            

            if run in train_runs:
                X_train.append(windows)
                y_train.append(y_binary)
            else:
                X_test.append(windows)
                y_test.append(y_binary)

            # print(f"Soggetto {subject} run {run} - Campioni: {X_csp.shape[0]}, Feature per campione: {X_csp.shape[1]}, Etichette: {y.shape[0]}")

        except Exception as e:
            print(f"Errore {subject}: {e}")

    X_train = np.concatenate(X_train, axis=0)
    y_train = np.concatenate(y_train)

    X_test = np.concatenate(X_test, axis=0)
    y_test = np.concatenate(y_test)

    pipe = Pipeline([
        ("csp", CSP(reg='ledoit_wolf')), # reg aiuta la stabilità con finestre corte
        ("scaler", StandardScaler()),    # Fondamentale per SVM
        ("svm", SVC(probability=True))
    ])

    param_grid = {
        "csp__n_components": [2, 4, 6],
        "csp__log": [True, False],
        "svm__C": [0.1, 1, 10, 100],
        "svm__gamma": ["scale"], 
        "svm__kernel": ["rbf", "linear"]
    }

    grid = GridSearchCV(
        pipe,
        param_grid,
        cv=5,
        scoring="balanced_accuracy",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)
    
    # Introduzione di logica probabilistica
    probs = grid.predict_proba(X_test)
    max_probs = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)

    threshold = 0.70
    accepted_mask = max_probs >= threshold

    accepted_predictions = predictions[accepted_mask]
    accepted_true_labels = y_test[accepted_mask]

    # Balanced misura un'accuratezza media tra le classi (così se ho 70% su una classe e 0 sull'altra ottengo comunque un valore adeguato)
    accuracy = balanced_accuracy_score(
        accepted_true_labels,
        accepted_predictions
    )
    print("----------------------------------------------------")
    print(f"Paziente {subject} - Run di test: {test_run}")
    # Coonfusion matrix: su etichette ridotte tramite threshold e su tutte le etichette
    cm = confusion_matrix(
        accepted_true_labels,
        accepted_predictions
    )
    # print(cm)
    cm = confusion_matrix(
        y_test,
        predictions
    )
    # print(cm)
    cm_sum += cm    # Calcolo la media delle matrici di confusione di tutti i soggetti (usando tutte le etichette)

    all_accuracy[i-first_person] = accuracy
    accepted = np.sum(accepted_mask)
    total = len(y_test)
    discarded = total - accepted

    print(f"Campioni totali/accettati/scartati: {total}/{accepted}/{discarded}")
    print(f"Percentuale scartata: {100*discarded/total:.2f}%")
    print(f"Balanced accuracy sui campioni accettati: {accuracy*100:.2f}%")


    # print("Best score:", grid.best_score_)
    # print("Best params:", grid.best_params_)
    # print("Parameter grid:", grid.param_grid)

    # test_accuracy = grid.score(X_test, y_test)
    # all_accuracy[i-first_person] = test_accuracy
    # print(f"Accuratezza reale su RUN 12 (Test Set): {test_accuracy:.4f}")

mid_accuracy = all_accuracy.mean()
cm_mean = cm_sum / people
print()
print(f"Accuratezza media: {all_accuracy.mean()}")
print(f"Matrice di confusione media: {cm_mean}")



----------------------------------------------------
Paziente 001 - Run di test: 4
Campioni totali/accettati/scartati: 101/48/53
Percentuale scartata: 52.48%
Balanced accuracy sui campioni accettati: 85.14%


C:\Users\papar\AppData\Roaming\Python\Python314\site-packages\sklearn\decomposition\_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
C:\Users\papar\AppData\Roaming\Python\Python314\site-packages\sklearn\decomposition\_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


----------------------------------------------------
Paziente 002 - Run di test: 4
Campioni totali/accettati/scartati: 101/44/57
Percentuale scartata: 56.44%
Balanced accuracy sui campioni accettati: 90.89%
----------------------------------------------------
Paziente 003 - Run di test: 4
Campioni totali/accettati/scartati: 101/40/61
Percentuale scartata: 60.40%
Balanced accuracy sui campioni accettati: 57.29%
----------------------------------------------------
Paziente 004 - Run di test: 4
Campioni totali/accettati/scartati: 101/72/29
Percentuale scartata: 28.71%
Balanced accuracy sui campioni accettati: 63.44%


C:\Users\papar\AppData\Roaming\Python\Python314\site-packages\sklearn\decomposition\_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(
C:\Users\papar\AppData\Roaming\Python\Python314\site-packages\sklearn\decomposition\_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


----------------------------------------------------
Paziente 005 - Run di test: 4
Campioni totali/accettati/scartati: 101/34/67
Percentuale scartata: 66.34%
Balanced accuracy sui campioni accettati: 28.60%


C:\Users\papar\AppData\Roaming\Python\Python314\site-packages\sklearn\decomposition\_fastica.py:132: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


----------------------------------------------------
Paziente 006 - Run di test: 4
Campioni totali/accettati/scartati: 101/70/31
Percentuale scartata: 30.69%
Balanced accuracy sui campioni accettati: 51.31%
----------------------------------------------------
Paziente 007 - Run di test: 4
Campioni totali/accettati/scartati: 101/76/25
Percentuale scartata: 24.75%
Balanced accuracy sui campioni accettati: 90.32%
----------------------------------------------------
Paziente 008 - Run di test: 4
Campioni totali/accettati/scartati: 101/47/54
Percentuale scartata: 53.47%
Balanced accuracy sui campioni accettati: 49.64%


KeyboardInterrupt: 